In [1]:
import pandas as pd
import numpy as np
import csv
import re

In [2]:
df = pd.read_excel(r"D:\НДВ\Города млн\Июль 2026\Первичка\Новая папка\Первичка-Млн_06-26.xlsx")

In [7]:
lower_words = {'на', 'в', 'и', 'с', 'к', 'у', 'о', 'от', 'по', 'за', 'из', 'над', 'под', 'об', 'же', 'ли', 'бы'}

def correct_title(text):
    if not isinstance(text, str):
        return text
    # Разбиваем на слова
    words = text.lower().split()
    # Первое слово всегда с большой буквы
    result = [words[0].capitalize()]
    # Остальные слова: если в списке исключений -> оставляем маленькими, иначе с большой
    for word in words[1:]:
        if word in lower_words:
            result.append(word)
        else:
            result.append(word.capitalize())
    return ' '.join(result)

df["Название проекта"] = df["Название проекта"].apply(correct_title)

In [8]:
df["Название проекта"].unique()

array(['17 Линия', '17/33 Петровский Остров', 'Amber Club', ...,
       'Аксаковский', 'Умный Дом', 'Новая Уфа'],
      shape=(1216,), dtype=object)

In [3]:
df2 = pd.read_excel(r"D:\НДВ\Города млн\Июль 2026\Города-миллионники_классы.xlsx")


In [9]:
df2["Название проекта"] = df2["Название проекта"].apply(correct_title)

In [10]:
df = df.merge(df2[['Название проекта', 'Локация2', 'Класс']],
                on=['Название проекта', 'Локация2'],
                how='left')

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240153 entries, 0 to 240152
Data columns (total 27 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Название проекта          240153 non-null  object 
 1   Девелопер                 240153 non-null  object 
 2   КлассOld                  216168 non-null  object 
 3   Локация                   240153 non-null  object 
 4   Локация2                  240153 non-null  object 
 5   Округ                     61227 non-null   object 
 6   Район                     201917 non-null  object 
 7   Микрорайон                95770 non-null   object 
 8   Метро                     104798 non-null  object 
 9   Улица                     114295 non-null  object 
 10  Дом                       105116 non-null  object 
 11  Корпус                    216168 non-null  object 
 12  Расстояние до центра, км  23985 non-null   float64
 13  Срок сдачи                239330 non-null  o

In [6]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 844 entries, 0 to 843
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         613 non-null    float64
 1   Название проекта           844 non-null    object 
 2   На англ                    165 non-null    object 
 3   Промзона                   696 non-null    object 
 4   Местоположение             817 non-null    object 
 5   Метро                      815 non-null    object 
 6   Расстояние до метро, км    815 non-null    object 
 7   Время до метро, мин        814 non-null    object 
 8   МЦК/МЦД                    815 non-null    object 
 9   Расстояние до МЦК/МЦД, км  814 non-null    object 
 10  Время до МЦК/МЦД, мин      809 non-null    object 
 11  БКЛ                        801 non-null    object 
 12  Расстояние до БКЛ, км      801 non-null    object 
 13  Время до БКЛ, мин          801 non-null    object 

In [14]:
df2['Название проекта'] = df2['Название проекта'].astype(str).str.strip().str.title()

In [21]:
df3 = pd.read_excel(r'C:\PycharmProjects\ndv_parcing\НашДомРФ\sales_agg.xlsx')

In [14]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 599 entries, 0 to 598
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              599 non-null    datetime64[ns]
 1   ID дом.рф         599 non-null    object        
 2   aps_perc          423 non-null    float64       
 3   aps_realised      477 non-null    float64       
 4   aps_total         477 non-null    float64       
 5   nonlive_perc      462 non-null    float64       
 6   nonlive_realised  477 non-null    float64       
 7   nonlive_total     477 non-null    float64       
 8   parking_perc      339 non-null    float64       
 9   parking_realised  477 non-null    float64       
 10  parking_total     477 non-null    float64       
 11  Название проекта  0 non-null      object        
 12  Девелопер         0 non-null      object        
dtypes: datetime64[ns](1), float64(9), object(3)
memory usage: 61.0+ KB


In [26]:
# 1. Приводим ID к строке и чистим
# Удаляем 'nan' (как строку), удаляем точки в конце
df['ID дом.рф'] = df['ID дом.рф'].astype(str).str.replace('nan', '', case=False).str.rstrip('.')
df3['ID дом.рф'] = df3['ID дом.рф'].astype(str).str.replace('nan', '', case=False).str.rstrip('.')

# 2. Удаляем пустые строки (если были 'nan')
df = df[df['ID дом.рф'] != '']
df3 = df3[df3['ID дом.рф'] != '']

# 3. Приводим к числовому виду для надежности (опционально)
# Это поможет отбросить нечисловые значения
df['ID дом.рф'] = pd.to_numeric(df['ID дом.рф'], errors='coerce')
df3['ID дом.рф'] = pd.to_numeric(df3['ID дом.рф'], errors='coerce')

# 4. Удаляем строки с NaN (если появились после преобразования)
df = df.dropna(subset=['ID дом.рф'])
df3 = df3.dropna(subset=['ID дом.рф'])

# 5. Приводим к целому числу (без .0) и затем к строке
df['ID дом.рф'] = df['ID дом.рф'].astype(int).astype(str)
df3['ID дом.рф'] = df3['ID дом.рф'].astype(int).astype(str)

# 6. Подтягиваем данные через merge
df3 = df3.merge(
    df[['ID дом.рф', 'Название проекта', 'Девелопер']],
    on='ID дом.рф',
    how='left'
)



In [27]:
df3.head()

,date,ID дом.рф,aps_perc,aps_realised,aps_total,nonlive_perc,nonlive_realised,nonlive_total,parking_perc,parking_realised,parking_total,Название проекта_x,Девелопер_x,Название проекта_y,Девелопер_y
0,2026-06-26,71592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Прокшино,А101
1,2026-06-26,71592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Прокшино,А101
2,2026-06-26,71592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Прокшино,А101
3,2026-06-26,71592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Прокшино,А101
4,2026-06-26,71592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Прокшино,А101


In [17]:
df['Название проекта'] = df['Название проекта'].astype(str).str.strip().str.title()

In [18]:
df1 = df.merge(
    df2[['Название проекта', 'Класс', 'Округ']],
    on='Название проекта',
    how='left'
)

In [19]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1455 entries, 0 to 1454
Data columns (total 34 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   ID дом.рф                                       1455 non-null   int64  
 1   Название проекта                                1455 non-null   object 
 2   Застройщик                                      1455 non-null   object 
 3   Группа компаний                                 1231 non-null   object 
 4   Статус                                          1455 non-null   object 
 5   Дата публикации проекта                         1455 non-null   object 
 6   Ввод в эксплуатацию                             1455 non-null   object 
 7   Выдача ключей                                   1455 non-null   object 
 8   Распроданность квартир                          1343 non-null   float64
 9   Остаток квартир                          

In [11]:
def add_location_column(df):
    df = df.copy()

    new_moscow = {'ТАО', 'НАО'}
    old_moscow = {'ЮАО', 'ЮВАО', 'ЗАО', 'ЮЗАО', 'СЗАО', 'СВАО', 'ВАО', 'САО', 'ЦАО', 'ЗелАО'}

    df["Локация"] = None

    df.loc[df["Округ"].isin(new_moscow), "Локация"] = "Новая Москва"
    df.loc[df["Округ"].isin(old_moscow), "Локация"] = "Старая Москва"

    # всё остальное, но не пустое
    df.loc[
        (~df["Округ"].isin(new_moscow | old_moscow)) & df["Округ"].notna(),
        "Локация"
    ] = "Московская область"

    return df

In [20]:
df1 = add_location_column(df1)

In [12]:
df.to_excel(r"D:\НДВ\Города млн\Июль 2026\Города-миллионники_Первичка.xlsx")